# Tool-Calling Agents & Agent Handoffs

Topics:
- Defining tools with `@tool` decorator
- `llm.bind_tools(tools)` — attaching tools to the LLM
- `ToolNode` — LangGraph's built-in tool executor
- The **agent loop**: `agent → tools → agent → ...`
- **Agent handoffs**: triage → route to specialist (sales/support/billing)
- Structured routing with `with_structured_output(HandoffDecision)`

**Tool-calling agent loop:**
```
START → agent_node → [conditional] → tools → agent_node (loop)
                  └─ no tool_calls ─► END
```

In [ ]:
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Defining Tools

The `@tool` decorator converts a regular Python function into a LangChain tool. The docstring becomes the tool description the LLM uses to decide when to call it.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression. Example: calculate('2 + 2')"""
    try:
        return f"The result of {expression} is {eval(expression)}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    data = {"new york": "72°F, Sunny", "london": "58°F, Cloudy", "tokyo": "68°F, Clear", "paris": "65°F, Partly Cloudy"}
    return data.get(city.lower(), f"Weather data not available for {city}")

@tool
def divide(a: float, b: float) -> str:
    """Divide two numbers."""
    if b == 0: return "Error: Division by zero"
    return f"{a} / {b} = {a/b}"

# Verify tools are properly decorated
print(f"Tool: {calculate.name}")
print(f"Description: {calculate.description}")

## 2. Building the Tool-Calling Agent

`llm.bind_tools(tools)` registers the tool schemas with the LLM. When invoked, the LLM may return `tool_calls` in its response — `ToolNode` executes them.

The conditional edge checks `last_message.tool_calls` — if present, go to `tools`; otherwise, `END`.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

tools = [calculate, get_weather, divide]
llm_with_tools = llm.bind_tools(tools)  # LLM now knows about these tools

def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> Literal["tools", "end"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"

# ToolNode automatically executes all tool_calls in the last AIMessage
tool_node = ToolNode(tools)

g = StateGraph(AgentState)
g.add_node("agent", agent_node)
g.add_node("tools", tool_node)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
g.add_edge("tools", "agent")  # loop back after tool execution

agent = g.compile()

# Single query
result = agent.invoke({"messages": [HumanMessage(content="What's 25 * 17?")]})
print(f"Answer: {result['messages'][-1].content}")
print(f"Total messages: {len(result['messages'])}")

## 3. Tool Execution Trace

Inspect every message in the result to see the full tool-calling chain.

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="Calculate 15% of 250 and check weather in Paris")]})

for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    print(f"[{i}] {msg_type}:")
    if isinstance(msg, HumanMessage):
        print(f"  {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → {tc['name']}({tc['args']})")
        else:
            print(f"  {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"  [{msg.name}] → {msg.content}")

## 4. Agent Handoffs

A **triage agent** analyzes the customer's message and routes to the appropriate specialist. The routing decision is **structured** (Pydantic model) so it's always parseable.

```
START → triage → [conditional] → sales   → END
                             ├─► support → END
                             ├─► billing → END
                             └─► END  (answered directly)
```

In [ ]:
class HandoffState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    current_agent: str
    handoff_reason: str
    context_summary: str

class HandoffDecision(BaseModel):
    handoff_to: Literal["sales", "support", "billing", "end"] = Field(description="Which agent to hand off to")
    reason: str = Field(description="Reason for handoff")
    context: str = Field(description="Key context to pass to next agent")

def triage_agent(state: HandoffState) -> dict:
    system = """You are a customer service triage agent. Route to:
    - sales: Product questions, purchases, upgrades
    - support: Technical issues, bugs, how-to
    - billing: Payments, invoices, refunds
    - end: Simple questions you can answer directly"""

    handoff_llm = llm.with_structured_output(HandoffDecision)
    decision = handoff_llm.invoke([SystemMessage(content=system)] + state["messages"])

    if decision.handoff_to == "end":
        resp = llm.invoke([SystemMessage(content="Provide a brief, helpful response."), *state["messages"]])
        return {"messages": [AIMessage(content=f"[Triage] {resp.content}")], "current_agent": "end"}

    return {
        "current_agent": decision.handoff_to,
        "handoff_reason": decision.reason,
        "context_summary": decision.context,
        "messages": [AIMessage(content=f"[Triage] Routing to {decision.handoff_to}: {decision.reason}")],
    }

def make_specialist(role: str, system: str):
    def specialist(state: HandoffState) -> dict:
        full_system = f"{system} Context from triage: {state.get('context_summary', 'None')}"
        resp = llm.invoke([SystemMessage(content=full_system), *state["messages"]])
        return {"messages": [AIMessage(content=f"[{role}] {resp.content}")], "current_agent": f"{role}_complete"}
    return specialist

def route_from_triage(state: HandoffState) -> str:
    return state["current_agent"] if state["current_agent"] in ["sales", "support", "billing"] else "end"

g = StateGraph(HandoffState)
g.add_node("triage",  triage_agent)
g.add_node("sales",   make_specialist("Sales",   "You are a sales specialist. Help with product questions."))
g.add_node("support", make_specialist("Support", "You are a technical support specialist. Be patient."))
g.add_node("billing", make_specialist("Billing", "You are a billing specialist. Be clear about policies."))

g.add_edge(START, "triage")
g.add_conditional_edges("triage", route_from_triage, {"sales": "sales", "support": "support", "billing": "billing", "end": END})
for node in ["sales", "support", "billing"]: g.add_edge(node, END)

service_agent = g.compile()

queries = [
    "My app keeps crashing when I try to upload photos",
    "I want to upgrade to the premium plan",
    "I was charged twice for my subscription",
    "What time do you close?",
]

for query in queries:
    print(f"Customer: {query}")
    result = service_agent.invoke({"messages": [HumanMessage(content=query)], "current_agent": "", "handoff_reason": "", "context_summary": ""})
    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            print(f"  {msg.content[:120]}")
    print()